In [ ]:
# Install dependencies (new Gemini SDK)
%pip install -q google-genai python-dotenv requests

In [ ]:
from google import genai

# CANNOT COMMIT THIS NOTEBOOK
GEMINI_KEY = "REDACTED_GEMINI_KEY"
GEMINI_MODEL = "gemini-3-flash-preview"

if not GEMINI_KEY or GEMINI_KEY == "PASTE_YOUR_GEMINI_KEY_HERE":
    raise ValueError("Set GEMINI_KEY in this cell before running.")

GENAI_CLIENT = genai.Client(api_key=GEMINI_KEY)

print("✅ Gemini configured")
print("✅ Model:", GEMINI_MODEL)
print("✅ GEMINI_KEY loaded:", bool(GEMINI_KEY))

# Optional: list available models
for m in GENAI_CLIENT.models.list():
    name = getattr(m, 'name', '')
    if 'flash' in name.lower():
        print(name)

In [ ]:
# Mount Google Drive + set a dedicated working folder for this project
USE_GOOGLE_DRIVE = True

from pathlib import Path

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        print('Google Drive mounted at /content/drive')
    except Exception as e:
        print('Drive mount skipped or unavailable:', e)

drive_root = Path('/content/drive/MyDrive')
DAP_HS_DIR = drive_root / 'dap-hs'
DAP_HS_DIR.mkdir(parents=True, exist_ok=True)

DRIVE_REPO_PATH = DAP_HS_DIR / '11th-DAP-hate-speech'
DRIVE_PARSED_FILE = DAP_HS_DIR / 'singlish_parsed.jsonl'
DRIVE_OUTPUT_FILE = DAP_HS_DIR / 'synthetic_dataset.jsonl'

if DRIVE_REPO_PATH.exists():
    PROJECT_ROOT = DRIVE_REPO_PATH
    print('Using PROJECT_ROOT from Drive repo:', PROJECT_ROOT)
else:
    PROJECT_ROOT = Path.cwd()
    print('No Drive repo folder found; using runtime PROJECT_ROOT:', PROJECT_ROOT)

print('DAP_HS_DIR:', DAP_HS_DIR)
print('DRIVE_PARSED_FILE:', DRIVE_PARSED_FILE, '| exists:', DRIVE_PARSED_FILE.exists())
print('DRIVE_OUTPUT_FILE:', DRIVE_OUTPUT_FILE)

In [ ]:
# Upload parsed file if missing
if not DRIVE_PARSED_FILE.exists():
    print('DRIVE_PARSED_FILE not found, please upload singlish_parsed.jsonl...')
    from google.colab import files
    uploaded = files.upload()
    if uploaded:
        first_name = next(iter(uploaded.keys()))
        Path('singlish_parsed.jsonl').write_bytes(uploaded[first_name])
        DRIVE_PARSED_FILE.write_bytes(Path('singlish_parsed.jsonl').read_bytes())
        print('Saved to:', DRIVE_PARSED_FILE)
    else:
        print('No file uploaded.')
else:
    print('Found parsed file at:', DRIVE_PARSED_FILE)